# Bölüm 8 — BÖLÜM 8: BİRLİKTELİK KURALLARI VE TAVSİYE SİSTEMLERİ

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 8. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q matplotlib mlxtend numpy pandas scikit-learn scikit-surprise scipy seaborn


## 8.1. Birliktelik Kuralları (Association Rules)


### Apriori Algoritmasının Adımları (Detaylı)

`bolum08/08_01_02_apriori-algoritmasinin-adimlari-2.txt`


In [ ]:
while Lₖ₋₁ ≠ ∅:
    Cₖ = apriori_gen(Lₖ₋₁)   # Join + Prune

# --- Algoritma: Apriori Pseudocode ---
return L₁ ∪ L₂ ∪ ... ∪ Lₖ   # Tüm sık öğe kümeleri


### Apriori Algoritmasının Adımları (Detaylı)

`bolum08/08_01_02_apriori-algoritmasinin-adimlari.txt`


In [ ]:
L₁ = {sık tekli öğe kümeleri}
k = 2

# --- Algoritma: Apriori Pseudocode ---
    for each transaction T in D:
        Cₜ = subset(Cₖ, T)    # T içindeki aday kümeleri bul
        for each candidate c in Cₜ:
            c.count++          # Destek sayacını artır

# --- Algoritma: Apriori Pseudocode ---
    Lₖ = {c ∈ Cₖ : c.count / |D| ≥ min_sup}
    k = k + 1


### Python Uygulaması — Kapsamlı Birliktelik Kuralı Analizi

`bolum08/08_01_02_python-uygulamasi-kapsamli-birliktelik-kurali-an.py`

_Kitap: Kod 8.1, Kod 8.2_


In [ ]:
import random
# --- Python: Veri Hazırlama ve One-Hot Encoding ---
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
import matplotlib.pyplot as plt
import time

# --- Python: Veri Hazırlama ve One-Hot Encoding ---
# ─── 1. Gerçekçi Market Sepeti Veri Seti ─────────────────────────
# 1000 işlemli sentetik market verisi oluştur
np.random.seed(42)

# --- Python: Veri Hazırlama ve One-Hot Encoding ---
urunler = ["Ekmek", "Süt", "Tereyağı", "Yumurta", "Peynir",
           "Zeytin", "Reçel", "Çay", "Kahve", "Bisküvi",
           "Makarna", "Domates Salçası", "Zeytinyağı", "Pirinç"]

# --- Python: Veri Hazırlama ve One-Hot Encoding ---
# Gerçekçi sepet kalıpları tanımla
patterns = [
    ["Ekmek", "Süt", "Tereyağı"],          # Kahvaltı seti
    ["Ekmek", "Yumurta", "Peynir", "Zeytin"],
    ["Makarna", "Domates Salçası", "Zeytinyağı"],  # Yemek seti
    ["Çay", "Bisküvi"],
    ["Kahve", "Süt"],
    ["Pirinç", "Domates Salçası"],
]

# --- Python: Veri Hazırlama ve One-Hot Encoding ---
dataset = []
for _ in range(1000):
    # Her işlem için 1-2 kalıp + rastgele ürünler
    sepet = set()
    for pattern in np.random.choice(len(patterns),
                                    size=np.random.randint(1, 3),
                                    replace=False):
        sepet.update(patterns[pattern])
    # Ek rastgele ürünler ekle (gürültü)
    n_extra = np.random.randint(0, 3)
    sepet.update(np.random.choice(urunler, n_extra, replace=False))
    dataset.append(list(sepet))

# --- Python: Veri Hazırlama ve One-Hot Encoding ---
# ─── 2. TransactionEncoder ile One-Hot Encoding ───────────────────
te = TransactionEncoder()
te_array = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_array, columns=te.columns_)

# --- Python: Veri Hazırlama ve One-Hot Encoding ---
print(f"Veri seti boyutu: {df.shape}")
print(f"Öğe sayısı: {len(te.columns_)}")
print(f"Ortalama sepet boyutu: {df.sum(axis=1).mean():.2f} ürün")
print("\nİlk 3 işlem:")
print(df.head(3).T)

# --- Python: Apriori vs FP-Growth Performans Karşılaştırması ---
# ─── 3. Apriori vs FP-Growth Karşılaştırması ─────────────────────
min_sup_values = [0.05, 0.08, 0.10, 0.15, 0.20]

# --- Python: Apriori vs FP-Growth Performans Karşılaştırması ---
results = []
for min_sup in min_sup_values:
    # Apriori
    t0 = time.time()
    fi_apriori = apriori(df, min_support=min_sup,
                         use_colnames=True, max_len=5)
    t_apriori = time.time() - t0

# --- Python: Apriori vs FP-Growth Performans Karşılaştırması ---
    # FP-Growth
    t0 = time.time()
    fi_fp = fpgrowth(df, min_support=min_sup,
                      use_colnames=True, max_len=5)
    t_fp = time.time() - t0

# --- Python: Apriori vs FP-Growth Performans Karşılaştırması ---
    results.append({
        "min_sup"       : min_sup,
        "fi_count"      : len(fi_fp),
        "t_apriori_ms"  : t_apriori * 1000,
        "t_fp_ms"       : t_fp * 1000,
    })

# --- Python: Apriori vs FP-Growth Performans Karşılaştırması ---
res_df = pd.DataFrame(results)
print("\n=== Apriori vs FP-Growth Performans Karşılaştırması ===")
print(f"{'min_sup':>8s} | {'Sık Küme':>9s} | {'Apriori(ms)':>12s} | {'FP-Growth(ms)':>14s}")
print("-" * 52)
for _, row in res_df.iterrows():
    print(f"{row.min_sup:>8.2f} | {row.fi_count:>9.0f} | {row.t_apriori_ms:>12.1f} | {row.t_fp_ms:>14.1f}")

# --- Python: Birliktelik Kuralları Çıkarma ve Tüm Metrikler ---
# ─── 4. FP-Growth ile Sık Öğe Kümesi Bulma ───────────────────────
frequent_itemsets = fpgrowth(
    df,
    min_support=0.07,      # En az %7 destek
    use_colnames=True,
    max_len=4)

# --- Python: Birliktelik Kuralları Çıkarma ve Tüm Metrikler ---
# Boyuta göre dağılım
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)
print("Sık öğe kümesi boyut dağılımı:")
print(frequent_itemsets["length"].value_counts().sort_index())

# --- Python: Birliktelik Kuralları Çıkarma ve Tüm Metrikler ---
# ─── 5. Birliktelik Kuralları Çıkarma (Tüm Metrikler) ─────────────
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5)

# --- Python: Birliktelik Kuralları Çıkarma ve Tüm Metrikler ---
# Ek metrikler hesapla
rules["leverage"] = rules["support"] - (
    rules["antecedent support"] * rules["consequent support"])

# --- Python: Birliktelik Kuralları Çıkarma ve Tüm Metrikler ---
# Conviction hesapla
rules["conviction"] = np.where(
    rules["confidence"] == 1.0,
    np.inf,
    (1 - rules["consequent support"]) / (1 - rules["confidence"]))

# --- Python: Birliktelik Kuralları Çıkarma ve Tüm Metrikler ---
print(f"\nToplam kural sayısı: {len(rules)}")
print("\n--- En İyi 10 Kural (Lift'e Göre) ---")
cols = ["antecedents", "consequents", "support",
        "confidence", "lift", "leverage", "conviction"]
print(rules[cols].sort_values("lift", ascending=False).head(10).to_string())

# --- Python: Birliktelik Kuralları Çıkarma ve Tüm Metrikler ---
# ─── 6. Kural Filtreleme Stratejileri ─────────────────────────────
# Strateji 1: Yüksek güven + yüksek lift
strong_rules = rules[
    (rules["confidence"] >= 0.7) &
    (rules["lift"] >= 1.5) &
    (rules["support"] >= 0.05)
].copy()
print(f"\nGüçlü Kural Sayısı (conf≥0.7, lift≥1.5): {len(strong_rules)}")

# --- Python: Birliktelik Kuralları Çıkarma ve Tüm Metrikler ---
# Strateji 2: Belirli bir ürün için öneriler (örn: Ekmek alanlara ne öner?)
ekmek_rules = rules[
    rules["antecedents"].apply(lambda x: "Ekmek" in str(x))
].sort_values("lift", ascending=False)
print("\nEkmek Alanlara Önerilen Ürünler:")
print(ekmek_rules[["antecedents","consequents","confidence","lift"]].head(5))

# --- Python: Görselleştirme — Scatter Plot ve Isı Haritası ---
# ─── 7. Kural Görselleştirmesi ────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# --- Python: Görselleştirme — Scatter Plot ve Isı Haritası ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Python: Görselleştirme — Scatter Plot ve Isı Haritası ---
# Scatter: Support vs Confidence, renk=Lift
sc = axes[0].scatter(
    rules["support"], rules["confidence"],
    c=rules["lift"], cmap="RdYlGn",
    s=rules["lift"] * 20, alpha=0.7)
plt.colorbar(sc, ax=axes[0], label="Lift")
axes[0].set_xlabel("Destek (Support)")
axes[0].set_ylabel("Güven (Confidence)")
axes[0].set_title("Birliktelik Kuralları: Support vs Confidence")
axes[0].axhline(y=0.7, color="red", linestyle="--", alpha=0.5, label="min_conf=0.7")
axes[0].legend()

# --- Python: Görselleştirme — Scatter Plot ve Isı Haritası ---
# Lift ısı haritası (tek ürün çiftleri)
single_ant = rules[rules["antecedents"].apply(lambda x: len(x)==1)]
single_both = single_ant[single_ant["consequents"].apply(lambda x: len(x)==1)]

# --- Python: Görselleştirme — Scatter Plot ve Isı Haritası ---
if len(single_both) > 0:
    pivot = single_both.pivot_table(
        values="lift",
        index=single_both["antecedents"].apply(lambda x: list(x)[0]),
        columns=single_both["consequents"].apply(lambda x: list(x)[0]),
        aggfunc="max")
    sns.heatmap(pivot, ax=axes[1], cmap="YlOrRd",
                annot=True, fmt=".2f", linewidths=0.5)
    axes[1].set_title("Lift Isı Haritası (Tekli Öğe Çiftleri)")

# --- Python: Görselleştirme — Scatter Plot ve Isı Haritası ---
plt.tight_layout()
plt.show()

# --- Python: Görselleştirme — Scatter Plot ve Isı Haritası ---
# ─── 8. Gerçek Veri: Online Retail Veri Seti (isteğe bağlı) ───────
# Büyük ölçekli gerçek veri için:
# url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
# df_retail = pd.read_excel(url)
# # Temizleme ve dönüşüm adımları...
# basket = df_retail.groupby(["InvoiceNo", "Description"])["Quantity"]\
#                   .sum().unstack().fillna(0)
# basket_bool = basket.applymap(lambda x: True if x > 0 else False)
# fi = fpgrowth(basket_bool, min_support=0.02, use_colnames=True)
# rules_retail = association_rules(fi, metric="lift", min_threshold=1.5)
# print(f"Kural sayısı: {len(rules_retail)}")


## 8.2. Tavsiye Sistemleri (Recommender Systems)


### 8.2.1.4. Python Uygulaması: TF-IDF ve Kosinüs Benzerliği

`bolum08/08_02_01_04_python-uygulamasi-tf-idf-ve-kosinus-benzerligi.py`


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Örnek Film Veri Seti
movies = pd.DataFrame({
    'film_adi': ['Matrix', 'Yıldızlararası', 'Inception', 'Titanik',
                  'Şaşkın Aşıklar', 'Dövüş Kulübü', 'Marslı', 'Forrest Gump'],
    'aciklama': [
        'yapay zeka simülasyon felsefe aksiyon bilim kurgu hacker',
        'uzay kara delik boyutlar aşk bilim kurgu zaman yolculuğu astronot',
        'rüya bilinçaltı psikoloji aksiyon bilim kurgu hırsız fikir',
        'aşk gemi trajedi tarih romantizm felaket okyanus',
        'romantik komedi aşk Türkiye İstanbul ilişki güldürü',
        'kimlik psikoloji şiddet toplum erkeklik çatışma gerilim',
        'uzay Mars astronot hayatta kalma bilim akılcı problem çözme',
        'saf aşk arkadaşlık tarih ABD Vietnam engel aşma drama',
    ],
    'tur': ['Bilim Kurgu/Aksiyon', 'Bilim Kurgu/Drama', 'Bilim Kurgu/Aksiyon',
             'Romantik/Drama', 'Romantik Komedi', 'Gerilim/Dram',
             'Bilim Kurgu/Drama', 'Drama']
})

# 2. TF-IDF Vektörizasyonu
tfidf = TfidfVectorizer(min_df=1, stop_words=None)
tfidf_matrix = tfidf.fit_transform(movies['aciklama'])
print(f'TF-IDF Matris boyutu: {tfidf_matrix.shape}')
# (8 film, X kelime) şeklinde bir matris oluşur

# 3. Kosinüs Benzerlik Matrisini hesapla
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print('Kosinüs Benzerlik Matrisi:')
sim_df = pd.DataFrame(cosine_sim, index=movies['film_adi'],
                       columns=movies['film_adi'])
print(sim_df.round(3))

# 4. Bir filme göre benzer filmleri tavsiye eden fonksiyon
def get_content_recommendations(film_adi, cosine_sim=cosine_sim,
                                  df=movies, n=3):
    """Verilen film adına göre en benzer N filmi döndürür."""
    idx = df[df['film_adi'] == film_adi].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]  # Kendisi hariç ilk n film
    film_indices = [i[0] for i in sim_scores]
    return df[['film_adi', 'tur']].iloc[film_indices].assign(
        benzerlik_skoru=[round(s[1], 4) for s in sim_scores])

# 5. Test: 'Matrix' filmine benzer filmler
print('\n=== Matrix izlediyseniz önerilir ===')
print(get_content_recommendations('Matrix', n=3))

print('\n=== Yıldızlararası izlediyseniz önerilir ===')
print(get_content_recommendations('Yıldızlararası', n=3))


### Python Uygulaması: Pearson Korelasyonu ile Öğe Bazlı Filtreleme

`bolum08/08_02_02_python-uygulamasi-pearson-korelasyonu-ile-oge-ba.py`


In [ ]:
import pandas as pd
import numpy as np

# 1. Kullanıcı-Film Puan Veri Seti
ratings_data = {
    'kullanici': ['Ali','Ali','Ali','Ali','Ali',
                  'Ayşe','Ayşe','Ayşe','Ayşe',
                  'Mehmet','Mehmet','Mehmet','Mehmet',
                  'Zeynep','Zeynep','Zeynep','Zeynep','Zeynep'],
    'film': ['Matrix','Yıldızlararası','Inception','Dövüş Kulübü','Marslı',
              'Matrix','Yıldızlararası','Inception','Marslı',
              'Matrix','Inception','Dövüş Kulübü','Titanik',
              'Matrix','Yıldızlararası','Inception','Marslı','Titanik'],
    'puan': [5, 4, 5, 4, 3,
             4, 5, 4, 4,
             5, 5, 3, 2,
             4, 4, 5, 3, 5]
}

df = pd.DataFrame(ratings_data)

# 2. Fayda Matrisi: Satırlar=Kullanıcılar, Sütunlar=Filmler
movie_matrix = df.pivot_table(index='kullanici',
                               columns='film', values='puan')
print('=== Fayda Matrisi ===')
print(movie_matrix)

# 3. Öğe Bazlı Benzerlik: 'corrwith' ile Pearson Korelasyonu
def get_item_based_recommendations(film_adi, movie_matrix, n=3):
    """
    Belirtilen film ile tüm diğer filmler arasındaki
    Pearson Korelasyonunu hesaplar ve en benzer n filmi döndürür.
    """
    film_ratings = movie_matrix[film_adi]
    # corrwith: film_ratings vektörü ile diğer tüm sütunlar arasındaki korelasyon
    similar = movie_matrix.corrwith(film_ratings)
    corr_df = pd.DataFrame(similar, columns=['Korelasyon'])
    corr_df = corr_df.dropna()
    corr_df = corr_df[corr_df.index != film_adi]  # Kendisi hariç
    return corr_df.sort_values(by='Korelasyon', ascending=False).head(n)

# 4. Test
print('\n=== Matrix izleyenler ne izler? ===')
print(get_item_based_recommendations('Matrix', movie_matrix))

print('\n=== Titanik izleyenler ne izler? ===')
print(get_item_based_recommendations('Titanik', movie_matrix))

# 5. Belirli bir kullanıcı için kişiselleştirilmiş öneri
def recommend_for_user(kullanici_adi, movie_matrix, n_sim=2, top_n=3):
    """
    Kullanıcının izlediği filmlere benzer,
    henüz izlemediği filmleri önerir.
    """
    user_ratings = movie_matrix.loc[kullanici_adi].dropna()
    watched = set(user_ratings.index)
    all_films = set(movie_matrix.columns)
    unwatched = all_films - watched

    scores = {}
    for film in watched:
        similar_films = movie_matrix.corrwith(movie_matrix[film])
        similar_films = similar_films.dropna()
        for sim_film, sim_score in similar_films.items():
            if sim_film in unwatched:
                if sim_film not in scores:
                    scores[sim_film] = 0
                scores[sim_film] += sim_score * user_ratings[film]

    return pd.Series(scores).sort_values(ascending=False).head(top_n)

print(f'\n=== Ayşe için Kişiselleştirilmiş Öneriler ===')
print(recommend_for_user('Ayşe', movie_matrix))


### Python Uygulaması: Scipy ile SVD Tabanlı Tavsiye Sistemi

`bolum08/08_02_03_04_python-uygulamasi-scipy-ile-svd-tabanli-tavsiye.py`

_Kitap: Kod 8.3_


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error
from math import sqrt

# 1. Kullanıcı-Film Seyrek Puan Matrisi (0 = izlenmemiş)
# Gerçek sistemlerde bu matris milyonlarca satırdan oluşur
ratings_matrix = np.array([
    # Matrix  Yıldızl Inception Dövüş  Titanik  Marslı  Forrest
    [   5,      4,       5,        4,       0,       3,       0   ],  # Ali
    [   4,      5,       4,        0,       0,       4,       0   ],  # Ayşe
    [   5,      0,       5,        3,       2,       0,       0   ],  # Mehmet
    [   4,      4,       5,        0,       5,       3,       4   ],  # Zeynep
    [   0,      2,       0,        0,       5,       0,       5   ],  # Selin
    [   3,      0,       4,        5,       0,       4,       0   ],  # Hasan
], dtype=float)

film_names = ['Matrix','Yıldızlararası','Inception','Dövüş Kulübü',
              'Titanik','Marslı','Forrest Gump']
user_names = ['Ali','Ayşe','Mehmet','Zeynep','Selin','Hasan']

print('=== Orijinal Seyrek Matris (0 = izlenmemiş) ===')
print(pd.DataFrame(ratings_matrix, index=user_names, columns=film_names))

# 2. Normalizasyon: Her kullanıcının ortalama puanını çıkar (bias correction)
# Yalnızca izlenmiş filmler üzerinden ortalama hesapla
ratings_masked = np.where(ratings_matrix == 0, np.nan, ratings_matrix)
user_means = np.nanmean(ratings_masked, axis=1)

# Ortalama farkını sadece izlenen filmlere uygula
R_demeaned = ratings_matrix.copy()
for i in range(len(user_means)):
    R_demeaned[i, ratings_matrix[i] != 0] -= user_means[i]

# 3. SVD Uygulaması - k=3 gizli faktör
k = 3  # Gizli faktör sayısı (hiperparametre)
U, sigma, Vt = svds(R_demeaned, k=k)
sigma_diag = np.diag(sigma)

print(f'\n=== SVD Sonuçları (k={k}) ===')
print(f'U (Kullanıcı-Faktör): {U.shape}')
print(f'Sigma: {sigma}')
print(f'Vt (Faktör-Film): {Vt.shape}')

# 4. Tahmin Matrisi oluştur (bias'ı geri ekle)
all_predicted = np.dot(np.dot(U, sigma_diag), Vt) + user_means.reshape(-1, 1)

pred_df = pd.DataFrame(
    np.round(all_predicted, 2),
    index=user_names,
    columns=film_names
)
print('\n=== SVD Tahmin Matrisi (Tüm boşluklar dolduruldu) ===')
print(pred_df)

# 5. Kişiselleştirilmiş Öneri Fonksiyonu
def recommend_svd(kullanici_adi, pred_df, original_matrix, top_n=3):
    """
    SVD tahmin matrisine göre, kullanıcının daha önce izlemediği
    en yüksek tahmini puana sahip filmleri önerir.
    """
    user_idx = user_names.index(kullanici_adi)
    user_preds = pred_df.loc[kullanici_adi].copy()
    # Orijinal matriste 0 olan (izlenmemiş) filmleri filtrele
    unwatched_mask = original_matrix[user_idx] == 0
    recommendations = user_preds[unwatched_mask].sort_values(ascending=False)
    return recommendations.head(top_n)

print('\n=== Selin için Kişiselleştirilmiş SVD Önerileri ===')
print(recommend_svd('Selin', pred_df, ratings_matrix))

print('\n=== Mehmet için Kişiselleştirilmiş SVD Önerileri ===')
print(recommend_svd('Mehmet', pred_df, ratings_matrix))

# 6. Model Değerlendirmesi: RMSE hesapla
# Sadece orijinal matriste dolu olan değerleri karşılaştır
actual = ratings_masked.flatten()
predicted = all_predicted.flatten()
mask = ~np.isnan(actual)  # izlenmiş filmler
rmse = sqrt(mean_squared_error(actual[mask], predicted[mask]))
print(f'\n=== Model Değerlendirmesi ===')
print(f'Eğitim Seti RMSE: {rmse:.4f}')
print('(0 = mükemmel tahmin; gerçek sistemlerde 0.85-1.10 aralığı hedeflenir)')


### 8.2.3.5. surprise Kütüphanesi ile Gelişmiş SVD

`bolum08/08_02_03_05_surprise-kutuphanesi-ile-gelismis-svd.py`

_Kitap: Kod 8.4_


In [ ]:
from surprise import SVD, SVDpp, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, train_test_split, GridSearchCV
import pandas as pd

# 1. Veri Hazırlama
ratings_dict = {
    'itemID': ['Matrix','Matrix','Matrix','Inception','Inception',
               'Yıldızlararası','Yıldızlararası','Titanik','Titanik','Marslı'],
    'userID': ['Ali','Ayşe','Mehmet','Ali','Zeynep',
               'Ayşe','Zeynep','Selin','Zeynep','Ali'],
    'rating': [5, 4, 5, 5, 5, 5, 4, 5, 5, 3]
}
df = pd.DataFrame(ratings_dict)

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['userID', 'itemID', 'rating']], reader)

# 2. SVD Modeli - 5 katlı çapraz doğrulama
svd_model = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)
results = cross_validate(svd_model, data, measures=['RMSE', 'MAE'],
                          cv=5, verbose=True)

print(f'Ortalama RMSE: {results["test_rmse"].mean():.4f}')
print(f'Ortalama MAE : {results["test_mae"].mean():.4f}')

# 3. SVD++ (daha gelişmiş: örtük geri bildirimi de kullanır)
svdpp_model = SVDpp(n_factors=20, n_epochs=20)
results_pp = cross_validate(svdpp_model, data, measures=['RMSE'], cv=3)
print(f'\nSVD++ Ortalama RMSE: {results_pp["test_rmse"].mean():.4f}')

# 4. Hiperparametre Optimizasyonu
param_grid = {
    'n_factors': [20, 50, 100],
    'n_epochs': [10, 20],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.1]
}
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3)
gs.fit(data)
print(f'\nEn iyi RMSE: {gs.best_score["rmse"]:.4f}')
print(f'En iyi parametreler: {gs.best_params["rmse"]}')

# 5. Eğitilmiş modelle tahmin üretme
trainset = data.build_full_trainset()
svd_model.fit(trainset)

# Zeynep'in Marslı'ya vereceği tahmini puanı hesapla
pred = svd_model.predict(uid='Zeynep', iid='Marslı', verbose=True)
print(f'\nZeynep - Marslı tahmini puan: {pred.est:.2f}')
